# Figure 7 — cross-cancer JUN-associated epithelial states

A/B follow `final/fig7_balance_cross_cancer_20261107/render_jun_only_symmetric.py`.
C/D/E/F follow the updated `panel_redraw_li/render_jun_only_y_axes.py`; they match
the four PDFs supplied in this directory. C/D retain left axis lines without y
tick marks; E/F retain left axis lines with y tick marks. Values, labels, colors,
limits and geometry are unchanged. This notebook contains data loading and
complete plotting code; upstream biological analyses remain in their original locations.


## Plot style


In [ ]:
from pathlib import Path
import json
import os
import xml.etree.ElementTree as ET

os.environ.setdefault('MPLCONFIGDIR', '/tmp/opencode/final_symmetric_fig7_matplotlib')
os.environ.setdefault('OMP_NUM_THREADS', '6')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '6')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib import colors as mcolors
from matplotlib import ft2font
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.text import Text
from scipy import stats

INK, WARM, PURPLE = '#000000', '#D26843', '#7B5EA7'
SLATE, CONTEXT = '#C4CBD2', '#DEE3E7'
PANEL_SIZE, TITLE_SIZE, SUBTITLE_SIZE, AXIS_SIZE, LABEL_SIZE = 13, 10, 9.5, 9, 8

LINEAGE_COLORS = {
    'Epithelial': '#B8C0C8', 'T_NK': '#B2A2C4', 'Myeloid': '#C3AB84',
    'Stromal': '#94B0A5', 'Endothelial': '#91B1C5', 'B / Plasma': '#C69AB0',
    'Mast': '#B5A18D', 'Schwann': '#AAA5BD', 'Other atlas cells': CONTEXT,
}

CRC_SUBTYPE_MARKERS = {
    'r06_05': 'IL32/CD3D', 'r06_04': 'GZMK/NKG7', 'r06_06': 'HMGB2/CD3D',
    'r06_00': 'GZMA/NKG7', 'r06_02': 'IL7R/CD3D',
}

LINEAGE_ORDER = list(LINEAGE_COLORS)

FONT_DIR = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813')
FONT_FILES = [FONT_DIR / name for name in ['Arial.TTF', 'Arialbd.TTF', 'Ariali.TTF', 'Arialbi.TTF']]
for font in FONT_FILES:
    font_manager.fontManager.addfont(str(font))
FONT = 'Arial'
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': [FONT],
    'font.size': LABEL_SIZE, 'axes.labelsize': AXIS_SIZE,
    'axes.titlesize': SUBTITLE_SIZE, 'axes.titleweight': 'normal', 'axes.titlelocation': 'center',
    'axes.linewidth': .75, 'xtick.labelsize': LABEL_SIZE, 'ytick.labelsize': LABEL_SIZE,
    'xtick.major.size': 2.8, 'ytick.major.size': 2.8,
    'xtick.major.width': .75, 'ytick.major.width': .75,
    'text.color': INK, 'axes.labelcolor': INK, 'axes.edgecolor': INK,
    'xtick.color': INK, 'ytick.color': INK, 'legend.labelcolor': INK,
    'legend.fontsize': LABEL_SIZE,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.grid': False,
    'text.usetex': False, 'text.parse_math': False,
    'mathtext.default': 'regular', 'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'mathtext.cal': 'Arial', 'mathtext.sf': 'Arial', 'mathtext.tt': 'Arial',
    'mathtext.fallback': None,
})


## Saved inputs and panel layout


In [ ]:
SOURCE = Path('/mnt/disk18t/lr_xcy/riku/kirc_205/final/fig7_balance_cross_cancer_20261107')
DATA = SOURCE / 'source_data'
JUN_DATA = SOURCE / 'source_data_jun_only'
# Preserve the parent main PDF. The assembled updated figure uses the redraw directory.
OUT = SOURCE / 'panel_redraw_li'
STEM = 'Fig7_JUN_only_symmetric'
DPI = 450
W, H, PW = 17.6, 11.85, 8.2
CLUSTERS = {'CRC': 'c08', 'PDAC': 'c7'}
TOP_SIX = {
    'CRC': ['c08', 'c01', 'c03', 'c05', 'c07', 'c09'],
    'PDAC': ['c7', 'c9', 'c5', 'c3', 'c6', 'c4'],
}
BOUNDS = {
    'A': (.40, .30, PW, 3.35), 'B': (9.00, .30, PW, 3.35),
    'C': (.40, 4.00, PW, 3.45), 'D': (9.00, 4.00, PW, 3.45),
    'E': (.40, 7.85, PW, 3.65), 'F': (9.00, 7.85, PW, 3.65),
}
PANEL_STEMS = {
    'A': 'Fig7A_CRC_JUN_only_joint_atlas',
    'B': 'Fig7B_PDAC_JUN_only_joint_atlas',
    'C': 'Fig7C_CRC_JUN_similarity_and_markers',
    'D': 'Fig7D_PDAC_JUN_similarity_and_markers',
    'E': 'Fig7E_CRC_JUN_CM1_cooccurrence',
    'F': 'Fig7F_PDAC_JUN_CM1_cooccurrence',
}
ATLAS, SIMILARITY, MARKERS, ASSOC = {}, {}, {}, {}
GEOMETRY, LINEAGE_ROWS, FITS, LEDGER = [], [], [], []
AXIS_CHECKS = []
COLLECT = True


def read(name, **kwargs):
    return pd.read_csv(DATA / name, float_precision='round_trip', **kwargs)


def record(element, source_file, selector, column, value, displayed=None):
    if not COLLECT:
        return
    if isinstance(value, np.generic):
        value = value.item()
    LEDGER.append(dict(figure_element=element, source_file=source_file,
                       source_selector=selector, source_column=column, value=value,
                       displayed=str(value) if displayed is None else displayed))


def load_data():
    global POINTS, COMPOSITION
    for cancer in ['CRC', 'PDAC']:
        sim = read(f'similarity/{cancer}_similarity.csv', usecols=[
            'cluster_id', 'cluster_label', 'eligible', 'n_cells', 'n_tumor_samples',
            'n_donors', 'n_studies', 'Sim_JUN', 'JUN_eligible_rank'])
        sim = sim[sim.eligible].copy().sort_values('Sim_JUN', ascending=False).reset_index(drop=True)
        chosen = sim.loc[sim.Sim_JUN.gt(0)].head(6).cluster_label
        sim['displayed'] = sim.cluster_label.isin(chosen)
        assert chosen.tolist() == TOP_SIX[cancer]
        SIMILARITY[cancer] = sim
        raw = read(f'similarity/{cancer}_expression_markers.csv')
        cluster = sim.loc[sim.cluster_label.eq(CLUSTERS[cancer]), 'cluster_id'].iloc[0]
        top = raw[raw.cluster_id.eq(cluster)].sort_values('rank').head(6).copy()
        metric = 'delta_log1p' if cancer == 'CRC' else 'donor_balanced_effect_vs_other_clusters'
        top['effect'] = top[metric]
        top['source_metric'] = metric
        assert len(top) == 6 and top.effect.gt(0).all()
        MARKERS[cancer] = top
    POINTS = read('E_balance_vs_CM1_per_sample.csv', usecols=[
        'cancer', 'sample_id', 'study_id', 'patient_key', 'n_epi', 'n_non_epi', 'JUN_fraction', 'CM1_activity'])
    source = read('E_F_single_cell_association_statistics.csv')
    for cancer in ['CRC', 'PDAC']:
        selected = source[source.cancer.eq(cancer) & source.unit.eq('sample')
                          & source.method.eq('raw') & source.feature.eq('JUN_fraction')]
        assert len(selected) == 1
        ASSOC[cancer] = selected.iloc[0]
    COMPOSITION = read('F_full_module_subtype_H_loadings.csv')


def load_atlas(cancer):
    if cancer not in ATLAS:
        prefix = 'C' if cancer == 'CRC' else 'D'
        ATLAS[cancer] = read(f'{prefix}_{cancer}_joint_atlas_per_cell_plot_data.csv.gz',
                             usecols=['UMAP1', 'UMAP2', 'broad_lineage', 'sample_id', 'is_JUN_pole'])
    return ATLAS[cancer]


## Shared drawing and display-fit functions


In [ ]:
def clean(ax):
    ax.spines[['top', 'right']].set_visible(False)
    for spine in ax.spines.values():
        spine.set_color(INK)
        spine.set_linewidth(.75)
    ax.tick_params(pad=4, labelsize=LABEL_SIZE, colors=INK)
    ax.grid(False)


def subplot_title(ax, title):
    return ax.set_title(title, loc='center', ha='center', color=INK,
                        fontsize=SUBTITLE_SIZE, fontweight='normal', pad=8)


def display_lineage(values):
    return values.replace({'B': 'B / Plasma', 'Plasma': 'B / Plasma',
                           'unassigned/mixed': 'Other atlas cells',
                           'unresolved/mixed': 'Other atlas cells'})


def padded_window(window, width, height):
    """Keep coordinate units undistorted and physical axis boxes identical."""
    cx, cy = (window.x_min + window.x_max) / 2, (window.y_min + window.y_max) / 2
    dx, dy = window.x_max - window.x_min, window.y_max - window.y_min
    if dx / dy < width / height:
        dx = dy * width / height
    else:
        dy = dx * height / width
    return (cx - dx / 2, cx + dx / 2), (cy - dy / 2, cy + dy / 2)


def pvalue(value):
    exponent = int(np.floor(np.log10(value)))
    # These authentic Arial TTFs lack U+2074/U+2075/U+2078/U+207B. Plain
    # scientific notation avoids both absent glyphs and MathText fallback.
    return f'{value / 10 ** exponent:.2f}e{exponent}'


def ols_display(x, y):
    """Unadjusted linear fit and pointwise t-based 95% conditional-mean CI."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    grid = np.linspace(x.min(), x.max(), 200)
    xc = x - x.mean()
    slope = np.dot(xc, y - y.mean()) / np.dot(xc, xc)
    intercept = y.mean() - slope * x.mean()
    fit = intercept + slope * grid
    residual_sd = np.sqrt(np.sum((y - intercept - slope * x) ** 2) / (len(x) - 2))
    se = residual_sd * np.sqrt(1 / len(x) + (grid - x.mean()) ** 2 / np.dot(xc, xc))
    band = stats.t.ppf(.975, len(x) - 2) * se
    return pd.DataFrame(dict(x=grid, fitted_CM1=fit, CI_low=fit - band, CI_high=fit + band))


def inspect_typography(fig):
    """Report the actual fonts, glyph coverage and visible text geometry."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    texts = list(fig.texts)
    titles = list(fig.texts[1::2])
    for ax in fig.axes:
        texts.extend(ax.texts)
        if ax.axison:
            titles.append(ax.title)
            texts.extend([ax.title, ax.xaxis.label, ax.yaxis.label])
            texts.extend(ax.get_xticklabels())
            texts.extend(ax.get_yticklabels())
        legend = ax.get_legend()
        if legend:
            texts.extend(legend.get_texts())
    texts = list({id(t): t for t in texts if t.get_visible() and t.get_text()}.values())
    titles = [t for t in titles if t.get_text()]
    font_paths, glyph_issues, colored_text, outside, boxes, overlaps = set(), [], [], [], [], []
    for text in texts:
        path = font_manager.findfont(text.get_fontproperties(), fallback_to_default=False)
        font_paths.add(path)
        font = ft2font.FT2Font(path)
        missing = sorted({char for char in text.get_text() if not char.isspace()
                          and not font.get_char_index(ord(char))})
        if missing:
            glyph_issues.append(dict(text=text.get_text(), missing=missing, font=path))
        if mcolors.to_hex(text.get_color()).upper() != INK:
            colored_text.append(text.get_text())
        box = Text.get_window_extent(text, renderer)
        if not all(fig.bbox.contains(*corner) for corner in box.get_points()):
            outside.append(text.get_text())
        boxes.append((text, box))
    for i, (text_a, box_a) in enumerate(boxes):
        for text_b, box_b in boxes[i + 1:]:
            # Ignore sub-pixel contacts between antialiased glyph bounds.
            width = min(box_a.x1, box_b.x1) - max(box_a.x0, box_b.x0)
            height = min(box_a.y1, box_b.y1) - max(box_a.y0, box_b.y0)
            if width > .7 and height > .7:
                overlaps.append([text_a.get_text(), text_b.get_text()])
    return dict(
        visible_text_count=len(texts), actual_font_files=sorted(font_paths),
        font_families=sorted({ft2font.FT2Font(path).family_name for path in font_paths}),
        font_sizes_pt=sorted({t.get_fontsize() for t in texts}),
        font_size_roles_pt=dict(panel_letters=PANEL_SIZE, panel_titles=TITLE_SIZE,
                                subplot_titles=SUBTITLE_SIZE, axis_titles=AXIS_SIZE,
                                tick_legend_cluster_and_statistical_text=LABEL_SIZE),
        title_count=len(titles), all_titles_centered=all(t.get_ha() == 'center' for t in titles),
        nonblack_text=colored_text, missing_native_glyphs=glyph_issues,
        text_outside_canvas=outside, overlapping_text_pairs=overlaps,
        mathtext_used=False, p_value_notation='plain scientific notation; no superscript glyph fallback',
        text_background_boxes=sum(t.get_bbox_patch() is not None for t in texts),
    )


## A/B — CRC and PDAC atlases


In [ ]:
class Panel:
    def __init__(self, fig, letter, x, y, w, h):
        self.fig, self.letter = fig, letter
        self.x, self.y, self.w, self.h = x, y, w, h
        self.fw, self.fh = fig.get_size_inches()
        self.axes = []

    def header(self, title):
        self.fig.text(self.x / self.fw, 1 - self.y / self.fh, self.letter,
                      fontsize=13, fontweight='bold', color=INK, ha='left', va='top')
        self.fig.text((self.x + self.w / 2) / self.fw, 1 - (self.y + .025) / self.fh,
                      title, fontsize=10, fontweight='bold', color=INK, ha='center', va='top')

    def ax(self, name, x, y, w, h):
        if COLLECT:
            GEOMETRY.append(dict(panel=self.letter, name=name, x=x, y=y, w=w, h=h))
        axis = self.fig.add_axes([(self.x + x) / self.fw, 1 - (self.y + y + h) / self.fh,
                                  w / self.fw, h / self.fh])
        self.axes.append((name, axis))
        return axis


def plot_atlas(ax, data, zoom=False):
    if zoom:
        ax.scatter(data.UMAP1, data.UMAP2, s=.58, c=CONTEXT, alpha=.72,
                   edgecolors='none', linewidths=0, rasterized=True, zorder=0)
    else:
        lineages = display_lineage(data.broad_lineage)
        order = ['Other atlas cells'] + [key for key in LINEAGE_ORDER if key != 'Other atlas cells']
        for lineage in order:
            d = data[lineages.eq(lineage)]
            if len(d):
                ax.scatter(d.UMAP1, d.UMAP2, s=.28, c=LINEAGE_COLORS[lineage], alpha=.83,
                           edgecolors='none', linewidths=0, rasterized=True, zorder=0)
    jun = data[data.is_JUN_pole]
    ax.scatter(jun.UMAP1, jun.UMAP2, s=1.05 if zoom else .86, c=WARM, alpha=.94,
               edgecolors='none', linewidths=0, rasterized=True, zorder=2)
    ax.set(xticks=[], yticks=[])
    for spine in ax.spines.values():
        spine.set_visible(zoom)
        spine.set_color(INK)
        spine.set_linewidth(.75)


def draw_atlas(p, cancer):
    prefix = 'C' if cancer == 'CRC' else 'D'
    data = load_atlas(cancer)
    expected_cells, expected_samples, expected_jun = (
        (643655, 316, 4855) if cancer == 'CRC' else (392388, 125, 3245))
    assert len(data) == expected_cells and data.sample_id.nunique() == expected_samples
    assert int(data.is_JUN_pole.sum()) == expected_jun
    p.header(f'{cancer} single-cell atlas')
    windows = read(f'{prefix}_{cancer}_view_windows.csv').set_index('view')
    full_x, full_y = padded_window(windows.loc['full'], 3.28, 2.60)
    zoom_x, zoom_y = padded_window(windows.loc['epithelial_zoom'], 1.80, 2.18)
    ax = p.ax('joint_atlas', .03, .55, 3.28, 2.60)
    plot_atlas(ax, data)
    ax.set(xlim=full_x, ylim=full_y)
    ax.add_patch(Rectangle((zoom_x[0], zoom_y[0]), zoom_x[1] - zoom_x[0], zoom_y[1] - zoom_y[0],
                           fill=False, ec=INK, lw=.75, ls=(0, (3.0, 2.7)), zorder=4))
    ax.plot([.06, .24], [.09, .09], transform=ax.transAxes, color=INK, lw=.75, clip_on=False)
    ax.plot([.06, .06], [.09, .31], transform=ax.transAxes, color=INK, lw=.75, clip_on=False)
    ax.text(.15, .035, 'UMAP 1', transform=ax.transAxes, fontsize=9,
            color=INK, va='top', ha='center')
    ax.text(.015, .20, 'UMAP 2', transform=ax.transAxes, fontsize=9,
            color=INK, va='center', ha='center', rotation=90)
    zoom = p.ax('epithelial_zoom', 3.49, .85, 1.80, 2.18)
    subplot_title(zoom, 'Epithelial zoom')
    inside = (data.UMAP1.between(*zoom_x) & data.UMAP2.between(*zoom_y)
              & data.broad_lineage.eq('Epithelial'))
    plot_atlas(zoom, data[inside], zoom=True)
    zoom.set(xlim=zoom_x, ylim=zoom_y)
    zoom.text(.12, .92, CLUSTERS[cancer], transform=zoom.transAxes,
              fontsize=8, color=INK, va='center', ha='center')

    counts = display_lineage(data.broad_lineage).value_counts()
    assert counts.sum() == expected_cells and counts.index.isin(LINEAGE_ORDER).all()
    handles, labels = [], []
    for lineage in LINEAGE_ORDER:
        value = int(counts.get(lineage, 0))
        name = {'T_NK': 'T/NK'}.get(lineage, lineage)
        handles.append(Line2D([], [], linestyle='none', marker='o', markersize=4.5,
                               markerfacecolor=LINEAGE_COLORS[lineage], markeredgecolor='none'))
        labels.append(f'{name} ({value:,})')
        if COLLECT:
            LINEAGE_ROWS.append(dict(cancer=cancer, lineage=lineage, n_cells=value,
                                     color=LINEAGE_COLORS[lineage]))
        record(f'{p.letter}.{cancer}.lineage.{lineage}',
               'source_data_jun_only/atlas_lineage_counts.csv',
               f'cancer={cancer};lineage={lineage}', 'n_cells', value, f'{value:,}')
    handles.append(Line2D([], [], linestyle='none', marker='o', markersize=4.5,
                           markerfacecolor=WARM, markeredgecolor='none'))
    labels.append(f'Epi_JUN-like: {CLUSTERS[cancer]} ({expected_jun:,})')
    record(f'{p.letter}.{cancer}.JUN_cells',
           f'source_data/{prefix}_{cancer}_frozen_poles.csv',
           'short_label=' + CLUSTERS[cancer], 'n_cells', expected_jun, f'{expected_jun:,}')
    key = p.ax('lineage_and_JUN_legend', 5.48, .64, 2.72, 2.55)
    key.set_axis_off()
    key.legend(handles, labels, loc='upper left', frameon=False, fontsize=8,
               labelcolor=INK, borderaxespad=0, handletextpad=.55, handlelength=.9, labelspacing=.69)


## C/D — JUN-reference similarity and selected-cluster markers


In [ ]:
def draw_similarity_and_markers(p, cancer):
    p.header(f'{cancer} epithelial similarity to ccRCC Epi_JUN')
    selected = SIMILARITY[cancer][SIMILARITY[cancer].displayed].sort_values('Sim_JUN', ascending=False)
    assert selected.cluster_label.tolist() == TOP_SIX[cancer]
    ax = p.ax('JUN_similarity', .62, .85, 3.10, 2.05)
    subplot_title(ax, 'Similarity to ccRCC Epi_JUN')
    xmax = .145 if cancer == 'CRC' else .39
    ticks = [0, .04, .08, .12] if cancer == 'CRC' else [0, .1, .2, .3]
    ax.set(xlim=(0, xmax), ylim=(5.65, -.65), yticks=range(6),
           yticklabels=selected.cluster_label, xticks=ticks)
    ax.set_xticklabels(['0'] + [f'{v:.2f}' if cancer == 'CRC' else f'{v:.1f}' for v in ticks[1:]])
    for i, row in enumerate(selected.itertuples()):
        color = WARM if row.cluster_label == CLUSTERS[cancer] else SLATE
        ax.plot([0, row.Sim_JUN], [i, i], color=color, lw=1.4, zorder=2)
        ax.scatter(row.Sim_JUN, i, s=38 if i == 0 else 28, c=color,
                   edgecolors='white', linewidths=.4, zorder=3)
        ax.text(row.Sim_JUN + xmax * .025, i, f'{row.Sim_JUN:.3f}',
                fontsize=8, color=INK, ha='left', va='center')
        record(f'{p.letter}.{cancer}.similarity.{row.cluster_label}',
               f'source_data/similarity/{cancer}_similarity.csv',
               'cluster_label=' + row.cluster_label, 'Sim_JUN', row.Sim_JUN, f'{row.Sim_JUN:.3f}')
    clean(ax)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0, pad=6)
    ax.set_xlabel('Sim_JUN', labelpad=6)

    markers = MARKERS[cancer]
    ax = p.ax('JUN_cluster_markers', 5.30, .85, 2.60, 2.05)
    subplot_title(ax, f'Top upregulated markers in {CLUSTERS[cancer]}')
    ax.barh(range(6), markers.effect, color=WARM, height=.55)
    ax.set(xlim=(0, 1.60), ylim=(5.65, -.65), yticks=range(6),
           yticklabels=markers.gene, xticks=[0, .5, 1, 1.5])
    for i, row in enumerate(markers.itertuples()):
        ax.text(row.effect + .035, i, f'{row.effect:.2f}', fontsize=8, color=INK, va='center')
        record(f'{p.letter}.{cancer}.marker.{row.gene}',
               f'source_data/similarity/{cancer}_expression_markers.csv',
               f'cluster_id={row.cluster_id};rank={row.rank};gene={row.gene}',
               row.source_metric, row.effect, f'{row.effect:.2f}')
    clean(ax)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0, pad=6)
    ax.set_xlabel('Expression difference', labelpad=6)


## E/F — JUN-like fraction, module activity and T/NK loadings


In [ ]:
def draw_cooccurrence(p, cancer):
    data = POINTS[POINTS.cancer.eq(cancer)]
    association = ASSOC[cancer]
    n = int(association.n)
    assert len(data) == n
    p.header(f'{cancer} co-occurrence with CM1-like (n = {n})')
    ax = p.ax('JUN_fraction_CM1_scatter', .62, .85, 3.10, 2.26)
    subplot_title(ax, 'Epi_JUN-like vs. CM1-like')
    ax.text(.035, .965, f'ρ = {association.rho:.3f}, P = {pvalue(association.p)}',
            transform=ax.transAxes, fontsize=8, color=INK, va='top', ha='left')
    fraction = 100 * data.JUN_fraction
    fit = ols_display(fraction, data.CM1_activity)
    ax.fill_between(fit.x, fit.CI_low, fit.CI_high, color=WARM, alpha=.15, lw=0, zorder=1)
    ax.plot(fit.x, fit.fitted_CM1, color=WARM, lw=1.1, zorder=2)
    ax.scatter(fraction, data.CM1_activity, s=22, c=WARM, alpha=.80,
               edgecolors='white', linewidths=.4, rasterized=True, zorder=3)
    ax.set(xlim=(-2, 65), xticks=[0, 20, 40, 60],
           ylim=(-.13, 3.60) if cancer == 'CRC' else (-.15, 1.85),
           yticks=[0, 1, 2, 3] if cancer == 'CRC' else [0, .5, 1, 1.5])
    ax.set_xlabel('Epi_JUN-like fraction (%)', labelpad=6)
    ax.set_ylabel('CM1-like activity', labelpad=6)
    clean(ax)
    if COLLECT:
        f = fit.copy()
        f.insert(0, 'cancer', cancer)
        FITS.append(f)
    for field in ['n', 'rho', 'p']:
        value = association[field]
        displayed = str(int(value)) if field == 'n' else f'{value:.3f}' if field == 'rho' else pvalue(value)
        record(f'{p.letter}.{cancer}.association.{field}',
               'source_data/E_F_single_cell_association_statistics.csv',
               f'cancer={cancer};unit=sample;method=raw;feature=JUN_fraction', field, value, displayed)

    comp = COMPOSITION[COMPOSITION.cancer.eq(cancer) & COMPOSITION.displayed_top5_TNK]
    comp = comp.sort_values('H_share', ascending=False)
    assert len(comp) == 5
    labels = []
    for subtype in comp.subtype:
        short = subtype.replace('CRCfa1TNK_', '')
        labels.append(f'{short} ({CRC_SUBTYPE_MARKERS[short]})' if cancer == 'CRC' else short)
    ax = p.ax('CM1_TNK_loadings', 5.30, .85, 2.60, 2.26)
    subplot_title(ax, 'CM1-like top T/NK loadings')
    ax.barh(range(5), 100 * comp.H_share, color=PURPLE, height=.52)
    ax.set(xlim=(0, 28), ylim=(4.65, -.65), yticks=range(5),
           yticklabels=labels, xticks=[0, 10, 20])
    for i, row in enumerate(comp.itertuples()):
        ax.text(100 * row.H_share + .7, i, f'{100 * row.H_share:.1f}%',
                fontsize=8, color=INK, va='center')
        record(f'{p.letter}.{cancer}.TNK.{row.subtype}',
               'source_data/F_full_module_subtype_H_loadings.csv',
               f'cancer={cancer};subtype={row.subtype}', 'H_share', row.H_share,
               f'{100 * row.H_share:.1f}%')
    clean(ax)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0, pad=6)
    ax.set_xlabel('Module loading (%)', labelpad=6)


## Updated C/D/E/F axes


In [ ]:
def draw_panel(fig, letter, bounds, collect_checks=False):
    panel = Panel(fig, letter, *bounds)
    DRAW[letter](panel)
    if letter in 'CDEF':
        for name, ax in panel.axes:
            bottom = ax.spines["bottom"]
            tick = ax.xaxis.get_major_ticks()[0].tick1line
            length, width = tick.get_markersize(), tick.get_markeredgewidth()
            ax.spines["left"].set_visible(True)
            ax.spines["left"].set_color(bottom.get_edgecolor())
            ax.spines["left"].set_linewidth(bottom.get_linewidth())
            ax.yaxis.set_ticks_position("left")
            show_y_ticks = letter in "EF"
            ax.tick_params(
                axis="y", which="both", left=show_y_ticks, right=False,
                labelleft=True, length=length if show_y_ticks else 0,
                width=width,
            )
            y_ticks = ax.yaxis.get_major_ticks() + ax.yaxis.get_minor_ticks()
            assert ax.spines["left"].get_visible()
            assert ax.spines["left"].get_linewidth() == bottom.get_linewidth()
            assert all(t.tick1line.get_visible() == show_y_ticks and not t.tick2line.get_visible() for t in y_ticks)
            assert all(t.tick1line.get_markersize() == (length if show_y_ticks else 0) for t in y_ticks)
            assert tick.get_markersize() == length and tick.get_markeredgewidth() == width
            if collect_checks:
                AXIS_CHECKS.append(dict(
                    panel=letter, axis=name, left_spine_visible=True,
                    spine_width_points=float(bottom.get_linewidth()),
                    y_tick_marks_visible=show_y_ticks,
                    y_tick_length_points=float(length) if show_y_ticks else 0,
                    y_labels_retained=True,
                    x_tick_length_points=float(length),
                    x_tick_width_points=float(width),
                ))
    return panel


DRAW = {
    'A': lambda p: draw_atlas(p, 'CRC'), 'B': lambda p: draw_atlas(p, 'PDAC'),
    'C': lambda p: draw_similarity_and_markers(p, 'CRC'),
    'D': lambda p: draw_similarity_and_markers(p, 'PDAC'),
    'E': lambda p: draw_cooccurrence(p, 'CRC'), 'F': lambda p: draw_cooccurrence(p, 'PDAC'),
}


## Export the assembled figure and individual panels


In [ ]:
def render_figure7(export_panels=True, export_main=True):
    """Render original A/B and updated C/D/E/F from the unchanged saved inputs.

    Standalone A/B keep their original output directory. Standalone C-F and the
    assembled updated figure use panel_redraw_li. No clustering, NMF, marker,
    similarity or Spearman analysis is rerun here.
    """
    global COLLECT
    for records in [GEOMETRY, LINEAGE_ROWS, FITS, LEDGER, AXIS_CHECKS]:
        records.clear()
    COLLECT = True
    load_data()
    fig = plt.figure(figsize=(W, H))
    for letter, bounds in BOUNDS.items():
        draw_panel(fig, letter, bounds, collect_checks=True)
    typography = inspect_typography(fig)
    assert not typography['text_outside_canvas']
    assert not typography['overlapping_text_pairs']
    assert not typography['missing_native_glyphs']
    expected = pd.read_csv(JUN_DATA / 'ALL_STATISTICS_SHOWN.csv', float_precision='round_trip')
    pd.testing.assert_frame_equal(pd.DataFrame(LEDGER), expected, check_dtype=False)
    if export_main:
        OUT.mkdir(parents=True, exist_ok=True)
        for ext in ['pdf', 'png', 'svg']:
            fig.savefig(OUT / f'{STEM}.{ext}', dpi=DPI)
        fig.savefig(OUT / f'{STEM}_preview.png', dpi=150)
    plt.close(fig)

    COLLECT = False
    if export_panels:
        for letter, (_, _, width, height) in BOUNDS.items():
            fig = plt.figure(figsize=(width + .20, height + .20))
            draw_panel(fig, letter, (.10, .10, width, height))
            panel_checks = inspect_typography(fig)
            assert not panel_checks['text_outside_canvas']
            assert not panel_checks['overlapping_text_pairs']
            panel_dir = (SOURCE if letter in 'AB' else OUT) / 'panels_jun_only'
            panel_dir.mkdir(parents=True, exist_ok=True)
            for ext in ['pdf', 'png', 'svg']:
                fig.savefig(panel_dir / f'{PANEL_STEMS[letter]}.{ext}', dpi=DPI)
            fig.savefig(panel_dir / f'Fig7{letter}_preview.png', dpi=170)
            plt.close(fig)
    COLLECT = True
    return dict(main_figure=str(OUT / f'{STEM}.pdf'),
                standalone_AB=str(SOURCE / 'panels_jun_only'),
                standalone_CDEF=str(OUT / 'panels_jun_only'),
                checked_numeric_records=len(LEDGER), axis_checks=AXIS_CHECKS.copy(),
                typography=typography)


In [ ]:
render_metadata = render_figure7()
render_metadata
